# Build network

Review the network topology inputs, estimate each substation's share of demand, and save a topology-only PyPSA network for the interruption analysis.

**Before running:** run `00_data_review.ipynb` first and use the `.venv` kernel. Optionally place GridFinder/OSM line files under `data/0-incoming/energy/`.

**Inputs** (from `00_data_review.ipynb`, under `data/1-processed/energy/provided`): `snapped_substations.parquet`, `transmission_routes.parquet`, `generation_register_template.csv`. For `source="base"` also supply reviewed `lines.csv` and `generators.csv`. Demand is attached later from `01_demand_settings.ipynb`.

The final cell saves `data/1-processed/energy/networks/<source>.nc` — `source="base"` for reviewed inputs, `source="inferred"` derived from OSM roads (the GridFinder approach).


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import pypsa

from mu_star_energy.distribution import build_service_weights
from mu_star_energy.network_source import build_network
from mu_star_energy.osm import OSMDownloadRequired, region_slug
from mu_star_energy.paths import incoming_energy_dir, processed_energy_dir

PROVIDED_DIR = processed_energy_dir() / "provided"
NETWORK_OUTPUT_DIR = processed_energy_dir() / "networks"
# Which network to build:
#   "base"     - reviewed lines.csv / generators.csv inputs.
#   "inferred" - distribution lines derived from OSM roads (the GridFinder
#                approach), used when reviewed inputs are unavailable.
NETWORK_SOURCE = "base"

# Region to build when NETWORK_SOURCE == "inferred" (ignored for "base").
# Required for "inferred": any OSM/Nominatim query, e.g. "Rodrigues, Mauritius".
# The shortcuts mauritius, rodrigues, agalega, st_brandon expand to full
# queries; mauritius is the main island. Roads and substations are fetched from
# OpenStreetMap and cached under data/0-incoming/energy/osm/<region>/.
OSM_REGION = None

# Output file stem under data/1-processed/energy/networks/. None uses "base" or
# "inferred-<region>"; set a name so a new build does not overwrite an old one.
OUTPUT_NAME = None

# Rebuild even if the output already exists. When False, an existing network is
# loaded and displayed instead of rebuilt.
OVERWRITE = False

# Permit an OpenStreetMap download when OSM_REGION is not cached yet. Left False
# so a run never downloads unexpectedly; set True for the first build.
ALLOW_DOWNLOAD = False

# OSM road detail for "inferred": "drive" (road network) or "all" (every way).
# "drive" is full enough for Mauritius.
OSM_NETWORK_TYPE = "drive"

substations = gpd.read_parquet(PROVIDED_DIR / "snapped_substations.parquet")
routes = gpd.read_parquet(PROVIDED_DIR / "transmission_routes.parquet")

generation_template = pd.read_csv(PROVIDED_DIR / "generation_register_template.csv")
generators_path = PROVIDED_DIR / "generators.csv"
generator_map_data = generation_template.copy()
if generators_path.exists():
    reviewed_generators = pd.read_csv(generators_path)
    if "generator_id" in reviewed_generators:
        reviewed_generators = reviewed_generators.drop_duplicates("generator_id").set_index("generator_id")
        if "capacity_mw" in reviewed_generators:
            reviewed_capacity = generator_map_data["generator_id"].map(reviewed_generators["capacity_mw"])
            generator_map_data["capacity_mw"] = reviewed_capacity.combine_first(
                generator_map_data["capacity_mw"]
            )

lines_path = PROVIDED_DIR / "lines.csv"
lines = pd.read_csv(lines_path) if lines_path.exists() else pd.DataFrame()

def format_unique_values(values, unit):
    numeric = pd.to_numeric(values, errors="coerce").dropna().unique()
    if not len(numeric):
        return ""
    joined = "/".join(f"{value:g}" for value in sorted(numeric))
    return f"{joined} {unit}"

## Transmission mapping

Plots the route geometry and snapped substations. Route labels show voltage/rating only where the route table or a linked reviewed line provides them. Most routes are unnamed and carry no endpoint, circuit-count, rating or status fields, so connecting lines are not drawn here.

In [ ]:
routes_for_plot = routes.copy()
if "v_nom_kv" not in routes_for_plot:
    routes_for_plot["v_nom_kv"] = pd.NA
if "capacity_mw" not in routes_for_plot:
    routes_for_plot["capacity_mw"] = pd.NA

routes_for_plot["voltage_label"] = routes_for_plot["v_nom_kv"].apply(
    lambda value: f"{value:g} kV" if pd.notna(value) else ""
)
routes_for_plot["rating_label"] = routes_for_plot["capacity_mw"].apply(
    lambda value: f"{value:g} MW" if pd.notna(value) else ""
)
line_ratings_mapped = 0
if not lines.empty and "source_route_id" in lines:
    for route_id, group in lines.groupby("source_route_id"):
        route_mask = routes_for_plot["route_id"].eq(str(route_id))
        voltage_label = format_unique_values(group.get("v_nom_kv", pd.Series(dtype=float)), "kV")
        rating_label = format_unique_values(group.get("s_nom_mva", pd.Series(dtype=float)), "MVA")
        if voltage_label:
            routes_for_plot.loc[route_mask, "voltage_label"] = voltage_label
        if rating_label:
            routes_for_plot.loc[route_mask, "rating_label"] = rating_label
            line_ratings_mapped += int(route_mask.any())

route_summary = routes_for_plot.assign(geometry_type=routes_for_plot.geometry.geom_type)[
    [
        "route_id",
        "name",
        "voltage_label",
        "rating_label",
        "length_km",
        "geometry_type",
    ]
]
display(route_summary.rename(columns={
    "route_id": "route ID",
    "name": "source name",
    "voltage_label": "voltage",
    "rating_label": "line rating",
    "length_km": "mapped geometry length (km)",
    "geometry_type": "geometry type",
}))

fig, ax = plt.subplots(figsize=(10, 10))
routes_for_plot.plot(
    ax=ax,
    color="#64748b",
    linewidth=2.2,
    alpha=0.75,
    label="transmission route",
)
for _, row in routes_for_plot.iterrows():
    rating_parts = []
    if row["voltage_label"]:
        rating_parts.append(row["voltage_label"])
    if row["rating_label"]:
        rating_parts.append(row["rating_label"])
    if not rating_parts:
        continue
    label_point = row.geometry.representative_point()
    ax.annotate(
        f"{row['route_id']}\n" + "\n".join(rating_parts),
        (label_point.x, label_point.y),
        xytext=(5, -12),
        textcoords="offset points",
        fontsize=6.5,
        color="#334155",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1},
    )
substations.plot(
    ax=ax,
    color="#ff8c00",
    edgecolor="black",
    markersize=50,
    label="substation",
)
for _, row in substations.iterrows():
    ax.annotate(
        row["bus_id"],
        (row.geometry.x, row.geometry.y),
        xytext=(3, 3),
        textcoords="offset points",
        fontsize=7,
    )
for generator_index, (_, row) in enumerate(generator_map_data.iterrows()):
    ax.scatter(
        row["lon"],
        row["lat"],
        marker="*",
        s=80,
        color="#16a34a",
        edgecolors="black",
        linewidths=0.5,
        zorder=7,
        label="generation site" if generator_index == 0 else None,
    )
    if pd.notna(row["capacity_mw"]):
        ax.annotate(
            f"{row['capacity_mw']:g} MW_e",
            (row["lon"], row["lat"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=6.5,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1},
            zorder=8,
        )
capacity_count = int(generator_map_data["capacity_mw"].notna().sum())
status_lines = [
    f"Generation capacities: {capacity_count}/{len(generator_map_data)} populated",
    f"Route voltage labels: {int(routes_for_plot['voltage_label'].astype(bool).sum())}/{len(routes_for_plot)} mapped",
    f"Route power-rating labels: {int(routes_for_plot['rating_label'].astype(bool).sum())}/{len(routes_for_plot)} mapped",
]
if not lines.empty and "source_route_id" not in lines:
    status_lines.append("Add source_route_id to lines.csv for map labels")
ax.text(
    0.01,
    0.01,
    "\n".join(status_lines),
    transform=ax.transAxes,
    fontsize=8,
    bbox={"facecolor": "white", "edgecolor": "#9ca3af", "alpha": 0.9},
)
ax.set_title("Network assets and available ratings")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2)
ax.legend()
plt.show()


## Estimating each substation's share of demand

OSM and GridFinder lines are optional. When available, the code assigns each mapped distribution-line section to the nearest transmission substation. The amount of mapped line near each substation is then used as a rough guide to its share of total demand.

This estimate is only for locating customers and demand. GridFinder routes are estimates and are not added to the electrical network calculation.


In [ ]:
def optional_line_layer(path):
    if not path.exists():
        return None
    return gpd.read_parquet(path) if path.suffix == ".parquet" else gpd.read_file(path)


gridfinder_path = incoming_energy_dir() / "gridfinder" / "grid.gpkg"
osm_path = incoming_energy_dir() / "osm" / "distribution_lines.parquet"
service_weights = build_service_weights(
    substations,
    gridfinder_lines=optional_line_layer(gridfinder_path),
    osm_distribution_lines=optional_line_layer(osm_path),
)
service_weights.to_csv(PROVIDED_DIR / "service_weights.csv", index=False)
display(service_weights.rename(columns={
    "bus_id": "substation ID",
    "gridfinder_km": "GridFinder line length (km)",
    "osm_km": "OSM line length (km)",
    "service_weight": "share of total demand",
    "method": "method used",
}))


## Information still needed

The reviewed `base` build needs `lines.csv` (line ID, endpoint substations, voltage, length and maximum power — not generated from the mapped route geometry) and `generators.csv`. Demand is attached during interruption analysis from a dated `demand_profile.csv` prepared in `01_demand_settings.ipynb`. The readiness table below shows what is present.


In [ ]:
generator_register = generator_map_data
required_line_columns = {"line_id", "bus0", "bus1", "v_nom_kv", "length_km", "s_nom_mva"}
line_columns_complete = required_line_columns.issubset(lines.columns)
line_values_complete = bool(
    line_columns_complete
    and len(lines)
    and lines[list(required_line_columns)].notna().all().all()
)

readiness = pd.Series({
    "vector route records": len(routes),
    "reviewed electrical line table present": int(lines_path.exists()),
    "reviewed electrical lines listed": len(lines),
    "line endpoint and rating fields complete": int(line_values_complete),
    "power stations with maximum output": int(generator_register["capacity_mw"].notna().sum()),
    "total power stations requiring maximum output": len(generator_register),
    "power stations with running cost": int(generator_register["marginal_cost"].notna().sum()),
    "total power stations requiring running cost": len(generator_register),
    "power stations with connected substation": int(generator_register["bus_id"].notna().sum()),
    "total power stations requiring a substation": len(generator_register),
    "demand-over-time file present for interruption run": int((PROVIDED_DIR / "demand_profile.csv").exists()),
    "method used to share demand": service_weights["method"].iloc[0] if len(service_weights) else "none",
})
display(readiness.to_frame("value"))


## Build and save the network

Builds and displays the selected network, saving it under `data/1-processed/energy/networks/`. `base` uses reviewed `lines.csv` and `generators.csv`. `inferred` needs an `OSM_REGION` and builds from that region's OSM roads (the GridFinder approach), saved as `inferred-<region>.nc`. An existing output is loaded instead of rebuilt unless `OVERWRITE = True`, and OpenStreetMap is only contacted when `ALLOW_DOWNLOAD = True`. Demand is attached later, during interruption analysis (see `01_demand_settings.ipynb`).


In [ ]:
# Resolve the output path first so an existing network is not rebuilt or
# overwritten by accident.
if OUTPUT_NAME:
    output_stem = OUTPUT_NAME
elif NETWORK_SOURCE == "inferred":
    output_stem = f"inferred-{region_slug(OSM_REGION)}" if OSM_REGION else "inferred"
else:
    output_stem = "base"
network_path = NETWORK_OUTPUT_DIR / f"{output_stem}.nc"

network = None
if network_path.exists() and not OVERWRITE:
    print(f"Loading existing network: {network_path}")
    print("Set OVERWRITE = True to rebuild it.")
    network = pypsa.Network(network_path)
else:
    try:
        outputs = build_network(
            NETWORK_SOURCE,
            input_dir=PROVIDED_DIR,
            output_dir=NETWORK_OUTPUT_DIR,
            region=OSM_REGION if NETWORK_SOURCE == "inferred" else None,
            output_name=OUTPUT_NAME,
            overwrite=OVERWRITE,
            allow_download=ALLOW_DOWNLOAD,
            network_type=OSM_NETWORK_TYPE,
            max_anchor_distance_m=1000,
        )
        network = pypsa.Network(outputs.network)
        print(f"Built and saved: {outputs.network}")
    except OSMDownloadRequired as needs_download:
        print(needs_download)
        print(
            "Building this region needs an OpenStreetMap download. "
            "Set ALLOW_DOWNLOAD = True, then re-run this cell."
        )
    except FileNotFoundError as missing_inputs:
        print("Missing inputs for the selected source:")
        print(missing_inputs)

if network is not None:
    display(
        pd.Series(
            {
                "output": output_stem,
                "buses": len(network.buses),
                "lines": len(network.lines),
                "generators": len(network.generators),
            },
            name="value",
        ).to_frame()
    )
    if {"x", "y"}.issubset(network.buses.columns) and len(network.buses):
        try:
            fig, ax = plt.subplots(figsize=(8, 8))
            for line in network.lines.itertuples():
                bus0 = network.buses.loc[line.bus0]
                bus1 = network.buses.loc[line.bus1]
                ax.plot([bus0.x, bus1.x], [bus0.y, bus1.y], color="#64748b", linewidth=0.8, zorder=1)
            ax.scatter(network.buses.x, network.buses.y, s=20, color="#ff8c00", edgecolors="black", linewidths=0.4, zorder=2)
            ax.set_title(f"{output_stem} network")
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            ax.grid(alpha=0.2)
            plt.show()
        except Exception as plot_error:
            print(f"(Skipped map: {plot_error})")


## How an outage is described

After the input checks are complete, mu-star supplies a table such as:

| `component` | `asset_id` | `available_fraction` |
|---|---|---:|
| Line | LINE_004 | 0.0 |
| Generator | Fort_George_1 | 0.4 |
| Bus | SUB_008 | 0.0 |

`component` identifies a line, power station (`Generator`) or substation (`Bus`, the PyPSA term). `available_fraction` is the usable share: 1 means fully available, 0 means completely unavailable and 0.4 means 40% available.

`EnergyModel.simulate(network, disruptions)` copies the provided fixed-capacity model, applies these reductions, recalculates electricity production and reports unmet demand, the share of demand served and operating cost. The separate damage work determines the available fraction provided to this model.
